# HVAC Takeoff Tool — YOLOv8 Training on Colab

Upload `yolo_dataset.zip` to Colab, then run all cells.

**GPU Runtime:** Go to Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics -q

In [ ]:
# Step 2: Upload and extract dataset
from google.colab import files
import zipfile, os

print('Upload yolo_dataset.zip...')
uploaded = files.upload()

with zipfile.ZipFile('yolo_dataset.zip', 'r') as z:
    z.extractall('/content/')

# Fix paths in dataset.yaml
import yaml
config_path = '/content/yolo_dataset/dataset.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)
config['path'] = '/content/yolo_dataset'
with open(config_path, 'w') as f:
    yaml.dump(config, f)

# Verify
train_imgs = len(os.listdir('/content/yolo_dataset/images/train'))
val_imgs = len(os.listdir('/content/yolo_dataset/images/val'))
print(f'Dataset: {train_imgs} train, {val_imgs} val images')
print(f'Classes: {config["names"]}')

In [ ]:
# Step 3: Train YOLOv8
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # small model — better accuracy than nano

results = model.train(
    data=config_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,          # GPU
    workers=2,
    project='/content/runs',
    name='hvac_detect',
    exist_ok=True,
    # Augmentation
    flipud=0.0,        # Don't flip vertically (blueprints have orientation)
    mosaic=0.5,        # Reduced mosaic (symbols need context)
    scale=0.3,         # Slight scale variation
)

print('\nTraining complete!')

In [ ]:
# Step 4: Check results
import pandas as pd

df = pd.read_csv('/content/runs/hvac_detect/results.csv')
df.columns = [c.strip() for c in df.columns]
best = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f"Best epoch: {int(best['epoch'])}")
print(f"Precision: {best['metrics/precision(B)']:.1%}")
print(f"Recall:    {best['metrics/recall(B)']:.1%}")
print(f"mAP50:     {best['metrics/mAP50(B)']:.1%}")
print(f"mAP50-95:  {best['metrics/mAP50-95(B)']:.1%}")

In [ ]:
# Step 5: Download the trained model
from google.colab import files
import shutil

# Copy best weights
shutil.copy('/content/runs/hvac_detect/weights/best.pt', '/content/hvac_yolov8s_v4.pt')
files.download('/content/hvac_yolov8s_v4.pt')
print('Download the .pt file and put it in hvac-takeoff-tool/models/')